In [19]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Benchmark shortlist (plan §5) — regression, not classification.
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Lasso, Ridge
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from catboost import CatBoostRegressor

# Pipelines. No SimpleImputer here: B_enriched has zero NaN by construction,
# which is precisely what the audit in the EDA notebook established.
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.preprocessing import OneHotEncoder, PolynomialFeatures, StandardScaler

# Splitting is grouped by country. A plain random CV would only re-measure the
# climate-fingerprint memorisation: (rain, temp) identifies the country exactly,
# so the same country on both sides of a split turns evaluation into a lookup.
from sklearn.model_selection import (
    GroupKFold,
    GroupShuffleSplit,
    RandomizedSearchCV,
    cross_validate,
    train_test_split,
)

# Metrics. `mean_squared_error(squared=False)` no longer exists in scikit-learn 1.9.
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.inspection import permutation_importance

import joblib
import mlflow
import mlflow.sklearn
from mlflow import MlflowClient
from mlflow.models import infer_signature

RANDOM_STATE = 42

# Chart style, identical to the EDA notebooks
BLUE = "#2b7bba"
INK = "#333333"

# <a id='toc2_'></a>[Chargement des données](#toc0_)

## <a id='toc2_1_'></a>[Lecture du fichier CSV](#toc0_)

In [20]:
df = pd.read_csv("df.csv")
df


,Area,Item,Year,hg/ha_yield,average_rain_fall_mm_per_year,pesticides_tonnes,avg_temp,cropland_1000ha,pesticides_kg_per_ha,yield_t_per_ha
0,Albania,Maize,1990,36613,1485.0,121.00,16.37,704.00,0.171875,3.6613
1,Albania,Maize,1991,29068,1485.0,121.00,15.36,703.00,0.172119,2.9068
2,Albania,Maize,1992,24876,1485.0,121.00,16.06,703.00,0.172119,2.4876
3,Albania,Maize,1993,24185,1485.0,121.00,16.05,702.00,0.172365,2.4185
4,Albania,Maize,1994,25848,1485.0,201.00,16.96,702.00,0.286325,2.5848
...,...,...,...,...,...,...,...,...,...,...
14366,Zimbabwe,Wheat,2009,30000,657.0,3269.99,20.52,4138.32,0.790173,3.0000
14367,Zimbabwe,Wheat,2010,27681,657.0,3305.17,21.17,4036.77,0.818766,2.7681
14368,Zimbabwe,Wheat,2011,26274,657.0,3340.35,20.78,4237.18,0.788343,2.6274
14369,Zimbabwe,Wheat,2012,24420,657.0,3375.53,20.52,4090.14,0.825285,2.4420


# <a id='toc3_'></a>[Utilitaires](#toc0_)

## <a id='toc3_1_'></a>[Fonctions de logging MLflow](#toc0_)

In [21]:
def log_model(model, name, X_sample):
    mlflow.sklearn.log_model(
        model,
        name=name,
        signature=infer_signature(X_sample, model.predict(X_sample)),
        input_example=X_sample.head(3),
    )

## <a id='toc3_2_'></a>[Gestion des runs](#toc0_)

In [22]:
def delete_run_if_exists(experiment_name, run_name):
    client = MlflowClient()
    exp = client.get_experiment_by_name(experiment_name)
    if exp is None:
        return
    runs = client.search_runs(
        experiment_ids=[exp.experiment_id],
        filter_string=f"tags.mlflow.runName = '{run_name}'",
    )
    for run in runs:
        client.delete_run(run.info.run_id)

## <a id='toc3_3_'></a>[Pipeline sklearn](#toc0_)

In [23]:
CAT = ["Item"]
NUM = ["Year", "average_rain_fall_mm_per_year", "avg_temp", "pest_pays_moyen", "pest_ecart"]

def encode(scale=False):
    """..."""
    onehot = ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CAT)
    if scale:
        # Modèles linéaires : chaque colonne est affectée, il ne reste rien.
        return ColumnTransformer([onehot, ("num", StandardScaler(), NUM)])
    # Arbres : le scaling ne sert à rien, mais il faut garder les numériques
    # explicitement — sinon ColumnTransformer les supprime.
    return ColumnTransformer([onehot], remainder="passthrough")


def build(*steps):
    """Chain steps into a pipeline that trains on log(yield) and predicts back in t/ha.

    Crops span 1.26 to 16 t/ha: on the raw scale the loss is dominated by potatoes.
    """
    return TransformedTargetRegressor(
        regressor=make_pipeline(*steps), func=np.log, inverse_func=np.exp
    )

# <a id='toc3_'></a>[Feature Engineering](#toc0_)

In [24]:
def clean_data(df):
    # Drop column: 'cropland_1000ha'
    df = df.drop(columns=['cropland_1000ha'])

    # Drop column: 'hg/ha_yield'
    df = df.drop(columns=['hg/ha_yield'])

    # Derive column 'pest_pays_moyen' from column: 'pesticides_kg_per_ha'
    df["pest_pays_moyen"] = df.groupby("Area")["pesticides_kg_per_ha"].transform("mean")

    # Derive column 'pest_ecart' from column: 'pesticides_kg_per_ha'
    df['pest_ecart'] = df['pesticides_kg_per_ha']-df['pest_pays_moyen']

    # Drop column: 'pesticides_kg_per_ha'
    df = df.drop(columns=['pesticides_kg_per_ha'])

    # for HistGradientBoosting, no need to encode the categorial variable if it is define a category type  
    df["Item"] = df["Item"].astype("category")

    return df

df_clean = clean_data(df.copy())
df_clean

,Area,Item,Year,average_rain_fall_mm_per_year,pesticides_tonnes,avg_temp,yield_t_per_ha,pest_pays_moyen,pest_ecart
0,Albania,Maize,1990,1485.0,121.00,16.37,3.6613,0.863920,-0.692045
1,Albania,Maize,1991,1485.0,121.00,15.36,2.9068,0.863920,-0.691801
2,Albania,Maize,1992,1485.0,121.00,16.06,2.4876,0.863920,-0.691801
3,Albania,Maize,1993,1485.0,121.00,16.05,2.4185,0.863920,-0.691556
4,Albania,Maize,1994,1485.0,201.00,16.96,2.5848,0.863920,-0.577596
...,...,...,...,...,...,...,...,...,...
14366,Zimbabwe,Wheat,2009,657.0,3269.99,20.52,3.0000,0.895431,-0.105258
14367,Zimbabwe,Wheat,2010,657.0,3305.17,21.17,2.7681,0.895431,-0.076665
14368,Zimbabwe,Wheat,2011,657.0,3340.35,20.78,2.6274,0.895431,-0.107089
14369,Zimbabwe,Wheat,2012,657.0,3375.53,20.52,2.4420,0.895431,-0.070147


# <a id='toc3_'></a>[Separation of sets](#toc0_)

In [25]:
X = df_clean[CAT + NUM]
y = df_clean["yield_t_per_ha"]



In [26]:
groups = df_clean["Area"]          # l'étiquette de groupe, PAS une feature de X

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
groups_train = groups.iloc[train_idx]


In [27]:
# --- Contrôles du split ---
groups_test = groups.iloc[test_idx]

print(f"train : {len(X_train):>6,} lignes ({len(X_train)/len(X):.0%})   {groups_train.nunique():>3} pays")
print(f"test  : {len(X_test):>6,} lignes ({len(X_test)/len(X):.0%})   {groups_test.nunique():>3} pays")

# Invariant : aucun pays des deux côtés — c'est toute la raison d'être du split groupé
fuite = set(groups_train) & set(groups_test)
assert not fuite, f"fuite de pays : {sorted(fuite)}"
print("aucun pays commun aux deux jeux : OK")

# Les 10 cultures doivent rester évaluables des deux côtés
cultures = pd.DataFrame({
    "train": X_train["Item"].value_counts(),
    "test": X_test["Item"].value_counts(),
}).fillna(0).astype(int)
manquantes = list(cultures.index[cultures["test"] == 0])
print(f"cultures absentes du test : {manquantes or 'aucune'}")
display(cultures)

# Ordres de grandeur de la cible
print(f"cible train : médiane {y_train.median():.2f}   moyenne {y_train.mean():.2f}   max {y_train.max():.1f}")
print(f"cible test  : médiane {y_test.median():.2f}   moyenne {y_test.mean():.2f}   max {y_test.max():.1f}")


train : 11,550 lignes (80%)    87 pays
test  :  2,821 lignes (20%)    22 pays
aucun pays commun aux deux jeux : OK
cultures absentes du test : aucune


,train,test
Item,,
Cassava,895,138
Maize,1719,488
Plantains and others,464,46
Potatoes,1792,478
"Rice, paddy",1362,299
Sorghum,1227,321
Soybeans,1180,241
Sweet potatoes,1025,253
Wheat,1524,442


cible train : médiane 3.95   moyenne 7.08   max 49.6
cible test  : médiane 4.12   moyenne 6.94   max 50.1


# <a id='toc5_'></a>[Configuration MLflow](#toc0_)

In [28]:
EXPERIMENT_NAME = "crop_yield_B"

# A run left open by an interrupted cell would silently swallow the next one.
if mlflow.active_run():
    mlflow.end_run()

mlflow.set_tracking_uri("http://127.0.0.1:5000/")
mlflow.set_experiment(EXPERIMENT_NAME)

2026/08/11 00:07:13 INFO mlflow.tracking.fluent: Experiment with name 'crop_yield_B' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1786399633739, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1786399633739, lifecycle_stage='active', name='crop_yield_B', tags={}, trace_location=None, workspace='default'>